# 3.11 图神经网络与大模型 (GNN + LLM)

> 🕐 预估学习时间：40分钟

知识图谱、引用网、仓库依赖、分子结构等天然是图。GNN+LLM 组合常见于 GraphRAG、工具路由、结构化推理与科学模型。

本节涵盖：
- 图消息传递基础
- 子图检索 + LLM 推理（GraphRAG 变体）
- 图编码器作为 LLM 工具/前缀
- 联合训练与评测要点


## 1. 消息传递 GNN（教学）

节点表示通过邻居聚合迭代更新：`h_v ← σ(W · mean({h_u : u∈N(v)} ∪ {h_v}))`。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


class MeanGCN(nn.Module):
    def __init__(self, d_in, d_hidden, n_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d_in if i == 0 else d_hidden, d_hidden) for i in range(n_layers)])

    def forward(self, x, adj):
        # adj: dense (N,N) with self-loops
        deg = adj.sum(-1, keepdim=True).clamp_min(1)
        for layer in self.layers:
            x = adj @ x / deg
            x = F.relu(layer(x))
        return x


# tiny KG: 0=Alice,1=Acme,2=Bob,3=Berlin
N, D = 4, 8
x = torch.randn(N, D)
adj = torch.tensor([
    [1, 1, 0, 0],  # Alice works_at Acme
    [1, 1, 1, 0],  # Acme related Bob
    [0, 1, 1, 1],  # Bob lives Berlin
    [0, 0, 1, 1],
], dtype=torch.float)
gcn = MeanGCN(D, 16)
node_h = gcn(x, adj)
print('=== GCN Node Embeddings ===')
print(node_h.shape, node_h.norm(dim=-1).tolist())
print(f'Key: GNN injects relational inductive bias before/with language reasoning.')


## 2. 子图检索 → 线性化为 LLM 上下文

对查询实体做 k-hop 扩展，把三元组序列化成文本或软提示。


In [ ]:
TRIPLES = [
    ('Alice', 'works_at', 'Acme'),
    ('Bob', 'works_at', 'Acme'),
    ('Bob', 'lives_in', 'Berlin'),
    ('Acme', 'located_in', 'Berlin'),
    ('Alice', 'knows', 'Bob'),
]


def retrieve_subgraph(seed: str, hops=2):
    frontier = {seed}
    got = []
    for _ in range(hops):
        new = set()
        for h, r, t in TRIPLES:
            if h in frontier or t in frontier:
                if (h, r, t) not in got:
                    got.append((h, r, t))
                new.add(h); new.add(t)
        frontier |= new
    return got


def linearize(triples):
    return '\n'.join(f'({h})-[{r}]->({t})' for h, r, t in triples)


sub = retrieve_subgraph('Alice', hops=2)
ctx = linearize(sub)
print('=== Subgraph Retrieval ===')
print(ctx)
prompt = f'Graph context:\n{ctx}\n\nQuestion: Where might Alice work geographically?'
print('prompt snippet:', prompt[:120], '...')
print(f'\nKey: GraphRAG retrieves structure, not just similar text chunks.')


## 3. 图向量作软前缀 / 工具

把种子节点嵌入投影为 LLM 前缀，或暴露 `graph.query` 工具让 Agent 主动查图。


In [ ]:
class GraphPrefix(nn.Module):
    def __init__(self, d_graph=16, d_llm=32, n_tokens=4):
        super().__init__()
        self.proj = nn.Linear(d_graph, n_tokens * d_llm)
        self.n_tokens = n_tokens
        self.d_llm = d_llm

    def forward(self, node_vec):
        return self.proj(node_vec).view(-1, self.n_tokens, self.d_llm)


class TinyLLM(nn.Module):
    def __init__(self, vocab=40, d=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.rnn = nn.GRU(d, d, batch_first=True)
        self.head = nn.Linear(d, vocab)

    def forward(self, tok, prefix=None):
        h = self.emb(tok)
        if prefix is not None:
            h = torch.cat([prefix, h], dim=1)
        y, _ = self.rnn(h)
        return self.head(y)


prefixer = GraphPrefix()
llm = TinyLLM()
# Alice node vec from GCN projected
alice_prefix = prefixer(node_h[0])
tok = torch.randint(0, 40, (1, 6))
logits = llm(tok, prefix=alice_prefix)
print('=== Graph Soft Prefix ===')
print(f'prefix={tuple(alice_prefix.shape)} logits={tuple(logits.shape)}')
print(f'Key: Soft prefixes keep graph geometry in continuous space; tools keep symbolic precision.')


## 4. 联合训练与评测

- 训练：对比学习对齐（文本↔子图）、或端到端 QA loss  
- 评测：Hit@k / MRR（链接预测）、图谱问答准确率、引用忠实度  
- 风险：图噪声放大幻觉；需证据三元组强制引用


In [ ]:
def mrr(ranks):
    return sum(1.0 / r for r in ranks) / len(ranks)


# toy: predicted ranking position of true tail
ranks = [1, 2, 1, 5, 3]
print('=== Graph+LLM Eval Toys ===')
print(f'MRR={mrr(ranks):.3f} Hit@3={sum(r<=3 for r in ranks)/len(ranks):.3f}')
print(f'\nKey: Report both structural retrieval metrics and end-task answer faithfulness.')


## 课后思考题

1. 何时用符号化三元组上下文，何时用软前缀？
2. 图检索的 hops 过大有何风险？如何剪枝？
3. GNN 与 LLM 分开训练 vs 联合训练的工程代价？
4. 如何防止模型忽略图证据只靠参数记忆作答？

---
> 本节涵盖了3.11 图神经网络与大模型的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
